In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration for **MAX LOAD TEST** ---
MAX_RECURSION_DEPTH = 10_000_000  # 🚀 🔥 Absolute limit test
OPTIMAL_DEPTH_STEP = 1_000_000  # 🔥 Extreme TPU recursion step
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 50_000_000  # 🔥 **50 MILLION** batch size

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    """🔥 Executes in extreme depth chunks to maximize TPU efficiency"""
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# --- TPU Sharding Setup ---
devices = jax.devices()
sharding = PositionalSharding(devices)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ **Extreme Recursive Execution**
def process_with_extreme_depths(x, total_depth):
    """🔥 Executes in **1M** depth chunks for TPU max stability"""
    iterations = total_depth // OPTIMAL_DEPTH_STEP
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP)
    return x

# --- Execute at MAXIMUM Performance ---
for depth in [1_000_000, 5_000_000, 10_000_000]:  # 🔥 **PUSH TPU TO THE EDGE**
    output_batch = process_with_extreme_depths(batch_input, depth)
    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)

NUM_TRIALS = 3  # 🔥 Reduce trials to avoid unnecessary TPU overload

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

for depth in [1_000_000, 5_000_000, 10_000_000]:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_extreme_depths(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

# --- Investigate TPU Compilation Stability ---
compiled_fn_1M = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=1_000_000)
compiled_fn_10M = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=10_000_000)

print("\n🚀 XLA Compilation for Depth=1,000,000:")
print(compiled_fn_1M.as_text())

print("\n🚀 XLA Compilation for Depth=10,000,000:")
print(compiled_fn_10M.as_text())



Batch Output Shape (Depth=1000000): (50000000,)
Batch Output Shape (Depth=5000000): (50000000,)
Batch Output Shape (Depth=10000000): (50000000,)


KeyboardInterrupt: 